Source:
https://www.datacamp.com/tutorial/knowledge-graph-rag
https://www.npmjs.com/package/download-git-repo

## PROCESS

1. Load files from repo
2. Create function chunking from each file
    Metadata:
        filename: ?
        line_number_mapping: ?
        function:
        function_call_stack
    Content:
3. Chunking
    TextSplitter
    https://python.langchain.com/docs/how_to/code_splitter/

STEP 2: Initialize language model
1. instantiate language model (OpenAI)
2. llm.transformer.convert_to_graph_documents

STEP 3: Store to vector database
Embeddings

STEP 4: Retrieve knowledge for RAG

Step 5: Evaluate response (langchain)

In [6]:
! pip install --upgrade pip

  Using cached pip-24.3.1-py3-none-any.whl.metadata (3.7 kB)
Using cached pip-24.3.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2


In [20]:
! pip install GitPython
! pip install langchain 
! pip install langchain-community
! pip install langchain_pinecone 
! pip install langchain-text-splitters
! pip install openai
! pip install os
! pip install pinecone
! pip install pinecone-client
! pip install pprint
! pip install pygithub 
! pip install python-dotenv
! pip install requests
! pip install streamlit
! pip install tiktoken
! pip install tree-sitter
! pip install tree-sitter-language-pack


  Using cached langchain_pinecone-0.0.1-py3-none-any.whl.metadata (1.5 kB)
INFO: pip is looking at multiple versions of langchain-pinecone to determine which version is compatible with other requirements. This could take a while.
ERROR: Ignored the following versions that require a different python version: 0.0.2 Requires-Python >=3.8.1,<3.13; 0.0.2rc0 Requires-Python >=3.8.1,<3.13; 0.0.3 Requires-Python >=3.8.1,<3.13; 0.1.0 Requires-Python <3.13,>=3.8.1; 0.1.1 Requires-Python <3.13,>=3.8.1; 0.1.2 Requires-Python <3.13,>=3.8.1; 0.1.3 Requires-Python <3.13,>=3.8.1; 0.2.0 Requires-Python <3.13,>=3.9; 0.2.0.dev1 Requires-Python <3.13,>=3.9
ERROR: Could not find a version that satisfies the requirement simsimd<4.0.0,>=3.6.3 (from langchain-pinecone) (from versions: 4.4.0, 5.0.0, 5.0.1, 5.1.0, 5.1.1, 5.1.2, 5.1.3, 5.1.4, 5.2.0, 5.2.1, 5.3.0, 5.4.0, 5.4.1, 5.4.2, 5.4.3, 5.4.4, 5.5.0, 5.5.1, 5.6.0, 5.6.1, 5.6.3, 5.6.4, 5.7.0, 5.7.1, 5.7.2, 5.7.3, 5.8.0, 5.9.0, 5.9.1, 5.9.2, 5.9.3, 5.9.4, 5.9.

# Constants

In [115]:

LANGUAGES_MAP = {
    '.py': 'python',
    '.js': 'javascript',
    '.jsx': 'javascript',
    '.java': 'java',
    '.cpp': 'cpp',
    '.ts': 'typescript',
    '.tsx': 'tsx',
    '.go': 'go',
    '.rs': 'rust',
    '.vue': 'vue',
    '.swift': 'swift',
    '.c': 'c',
    '.h': 'c',
    '.ipynb': 'ipynb',
}

SUPPORTED_EXTENSIONS = LANGUAGES_MAP.keys()

IGNORED_DIRS = {
    'node_modules', 
    'venv', 
    'env', 
    'dist', 
    'build', 
    'vendor',
    '__pycache__', 
    'pacakge.json',
    'tsconfig.json'
}

MAX_TOKENS_CHUNK_SIZE = 5000

# Setting API keys or secrets

In [99]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()

pinecone_api_key = os.getenv('PINECONE_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')

# Clone a github repo locally

In [ ]:
import re
import os
import requests
from git import Repo, GitCommandError
from pprint import pprint
from typing import Any, Dict, Generator, Tuple
from langchain.schema import Document


class GithubRepoData:
    def __init__(self, url: str):
        """
        Initializes the GithubRepoData object with repository details.

        Args:
            url (str): The GitHub repository URL.
            dir (str): The local directory to clone the repository into. Defaults to the current directory.
        """
        # Match the URL to extract the owner and repo name
        pattern = r"(?:https?://|git@)github\.com[:/](.+?)/(.+?)(?:/|\.git|$)"
        match = re.match(pattern, url)
        if not match:
            raise ValueError(f"Invalid GitHub URL: {url}")
        owner, repo = match.groups()

        # Fetch repository details from GitHub API
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        response = requests.get(api_url)
        if response.status_code == 200:
            github = response.json()
            visibility = github.get('visibility', 'unknown')

            if visibility == 'public':
                self.id = github.get('id')
                self.owner = github.get('owner').get('login')
                self.fullname = github.get('full_name')
                self.repo_name = github.get('name')
                self.main_branch = github.get('default_branch')
                self.description = github.get('description')
                self.events_url = github.get('events_url')
                # Set the directory path to the current directory if `dir` is not provided
                self.dir_path = os.path.join(self.repo_name)
            else:
                raise ValueError(f'Repository {self.owner}/{self.repo_name} is not public.')
        elif response.status_code == 404:
            raise ValueError(f"Repository {self.owner}/{self.repo_name} not found.")
        elif response.status_code == 403:
            raise ValueError("GitHub API rate limit exceeded. Please try again later.")
        else:
            raise ValueError(f"Failed to fetch repository details for {self.owner}/{self.repo_name}. HTTP Status: {response.status_code}")

    def to_dict(self):
        return {
            "id": self.id,
            "owner": self.owner,
            "fullname": self.fullname,
            "repo_name": self.repo_name,
            "description": self.description,
            "events_url": self.events_url,
            "dir_path": self.dir_path,
            "url": f"https://github.com/{self.owner}/{self.repo_name}"
        }
    
    def exists(self) -> bool:
        """
        Checks if the repository is already cloned.

        Returns:
            bool: True if the repository is already cloned, False otherwise.
        """
        return os.path.exists(self.dir_path)

    def clone(self) -> bool:
        """
        Clones the repository into the specified local directory.

        Returns:
            bool: True if the cloning is successful.

        Raises:
            ValueError: If cloning fails.
        """
        if self.exists():
            print(f"Repository {self.repo_name} already cloned at {self.dir_path}.")
            return True

        # Create the local directory if it doesn't exist
        if not os.path.exists(self.dir_path):
            os.makedirs(self.dir_path, exist_ok=True)
        print(f'Cloning {self.repo_name} into {self.dir_path}')

        # Clone the repository
        try:
            clone_url = f"https://github.com/{self.fullname}.git"
            Repo.clone_from(clone_url, self.dir_path)
            print(f"Repository {self.repo_name} cloned successfully.")
            return True
        except GitCommandError as e:
            raise ValueError(f'Failed to clone {self.repo_name}: {e}')
        
    def read_file(self, filepath):
        """Reads the contents of a file in the repository."""
        try:
            with open(filepath, "r", encoding='utf-8') as f:
                contents = f.read()
            return contents
        except Exception as e:
            raise ValueError(f"Failed to read file {filepath}: {e}")

    def get_files(self):
            """
            Recursively retrieves all files in `self.dir_name`, filtered by `supported_extensions` and skipping directories
            listed in `ignored_dirs`.

            Yields:
                str: Relative file paths for files that match the criteria.
            """
            documents = []
            for root, _, files in os.walk(self.dir_path):
                
                # Skip if current directory is in ignored directories
                if any(ignored_dir in root for ignored_dir in IGNORED_DIRS):
                    continue

                # Process each file in current directory
                for file in files:
                    file_path = os.path.join(root, file)
                    if os.path.splitext(file)[1] in SUPPORTED_EXTENSIONS:
                        content = self.read_file(file_path)
                        gitFile = file_path.removeprefix(self.repo_name +'/')
                        if content:
                            document = Document(
                                page_content = content,
                                metadata={
                                    "filename": file,
                                    "path": file_path,
                                    "url": f"https://www.github.com/{self.fullname}/blob/{self.main_branch}/{gitFile}",
                                }
                            )
                            documents.append(document)
                    else:
                        os.remove(file_path)
            return documents


In [108]:
repo = GithubRepoData("https://github.com/CoderAgent/SecureAgent")
# repo = GithubRepoData('https://github.com/itancio/braintumor2')
if not repo.exists():
    repo.clone()
files = repo.get_files()
for file in files:
    print(file)

page_content='import { Octokit } from "@octokit/rest";
import { createNodeMiddleware } from "@octokit/webhooks";
import { WebhookEventMap } from "@octokit/webhooks-definitions/schema";
import * as http from "http";
import { App } from "octokit";
import { Review } from "./constants";
import { env } from "./env";
import { processPullRequest } from "./review-agent";
import { applyReview } from "./reviews";

// This creates a new instance of the Octokit App class.
const reviewApp = new App({
  appId: env.GITHUB_APP_ID,
  privateKey: env.GITHUB_PRIVATE_KEY,
  webhooks: {
    secret: env.GITHUB_WEBHOOK_SECRET,
  },
});

const getChangesPerFile = async (payload: WebhookEventMap["pull_request"]) => {
  try {
    const octokit = await reviewApp.getInstallationOctokit(
      payload.installation.id
    );
    const { data: files } = await octokit.rest.pulls.listFiles({
      owner: payload.repository.owner.login,
      repo: payload.repository.name,
      pull_number: payload.pull_request.number

# Parser/ Chunker

In [464]:
from tiktoken import get_encoding

# Initialize tokenizer using the latest tokenizer
tokenizer = get_encoding('o200k_base')

def add_lineNumbers(content: str):
    contents_dict = {}
    for line_number, line in enumerate(content.split('\n'), start=1):
        contents_dict[line_number] = line
    return contents_dict

for file in files:
    numbered = add_lineNumbers(file['content'])
    pprint(numbered)

{1: 'import { Octokit } from "@octokit/rest";',
 2: 'import { createNodeMiddleware } from "@octokit/webhooks";',
 3: 'import { WebhookEventMap } from "@octokit/webhooks-definitions/schema";',
 4: 'import * as http from "http";',
 5: 'import { App } from "octokit";',
 6: 'import { Review } from "./constants";',
 7: 'import { env } from "./env";',
 8: 'import { processPullRequest } from "./review-agent";',
 9: 'import { applyReview } from "./reviews";',
 10: '',
 11: '// This creates a new instance of the Octokit App class.',
 12: 'const reviewApp = new App({',
 13: '  appId: env.GITHUB_APP_ID,',
 14: '  privateKey: env.GITHUB_PRIVATE_KEY,',
 15: '  webhooks: {',
 16: '    secret: env.GITHUB_WEBHOOK_SECRET,',
 17: '  },',
 18: '});',
 19: '',
 20: 'const getChangesPerFile = async (payload: '
     'WebhookEventMap["pull_request"]) => {',
 21: '  try {',
 22: '    const octokit = await reviewApp.getInstallationOctokit(',
 23: '      payload.installation.id',
 24: '    );',
 25: '    const 

In [ ]:
from tiktoken import get_encoding

class FileChunker:
    # Initialize tokenizer using the latest tokenizer
    tokenizer = get_encoding('o200k_base')
    def __init__(self): 
        self.num_tokens = 0
    def count_tokens(self, text):
        encoded_text = tokenizer.encode(text, disallowed_special=())
        return len(encoded_text)
    def chunk(self, file):
        filename_tokens_count = self.count_tokens(file.name)
        remaining_tokens = MAX_TOKENS - filename_tokens_count
        for line_number, line in file.content:
            line_tokens_count = self.count_tokens(line)
            if remaining_tokens >= line_tokens_count:
                remaining_tokens -= line_tokens_count
                yield line_number, line
            else:
                break

        

    def initialize_parser(file):
        filename = file.name
        extension = os.path.splitext(filename)[1]
        language = LANGUAGES_MAP[extension]
        return AbstractParser(language)

    filename = 'sample.py'
    parser = initialize_parser(filename)
    ast = parser.findingEnclosingContext(file, lineStart, lineEnd)

AttributeError: 'str' object has no attribute 'name'

In [524]:
! pip install nbformat semchunk

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)


In [526]:

"""Chunker abstraction and implementations."""

import logging
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass
from functools import cached_property
from typing import Any, Dict, List, Optional

import nbformat
import pygments
import tiktoken
from semchunk import chunk as chunk_via_semchunk
from tree_sitter import Node
from tree_sitter_language_pack import get_parser


tokenizer = tiktoken.get_encoding("cl100k_base")


class Chunk:
    @abstractmethod
    def content(self) -> str:
        """The content of the chunk to be indexed."""

    @abstractmethod
    def metadata(self) -> Dict:
        """Metadata for the chunk to be indexed."""


@dataclass
class FileChunk(Chunk):
    """A chunk of code or text extracted from a file in the repository."""

    file_content: str  # The content of the entire file, not just this chunk.
    file_metadata: Dict  # Metadata of the entire file, not just this chunk.
    start_byte: int
    end_byte: int

    @cached_property
    def filename(self):
        if not "file_path" in self.file_metadata:
            raise ValueError("file_metadata must contain a 'file_path' key.")
        return self.file_metadata["file_path"]

    @cached_property
    def content(self) -> Optional[str]:
        """The text content to be embedded. Might contain information beyond just the text snippet from the file."""
        return self.filename + "\n\n" + self.file_content[self.start_byte : self.end_byte]

    @cached_property
    def metadata(self):
        """Converts the chunk to a dictionary that can be passed to a vector store."""
        # Some vector stores require the IDs to be ASCII.
        filename_ascii = self.filename.encode("ascii", "ignore").decode("ascii")
        chunk_metadata = {
            # Some vector stores require the IDs to be ASCII.
            "id": f"{filename_ascii}_{self.start_byte}_{self.end_byte}",
            "start_byte": self.start_byte,
            "end_byte": self.end_byte,
            "length": self.end_byte - self.start_byte,
            # Note to developer: When choosing a large chunk size, you might exceed the vector store's metadata
            # size limit. In that case, you can simply store the start/end bytes above, and fetch the content
            # directly from the repository when needed.
            TEXT_FIELD: self.content,
        }
        chunk_metadata.update(self.file_metadata)
        return chunk_metadata

    @cached_property
    def num_tokens(self):
        """Number of tokens in this chunk."""
        return len(tokenizer.encode(self.content, disallowed_special=()))

    def __eq__(self, other):
        if isinstance(other, Chunk):
            return (
                self.filename == other.filename
                and self.start_byte == other.start_byte
                and self.end_byte == other.end_byte
            )
        return False

    def __hash__(self):
        return hash((self.filename, self.start_byte, self.end_byte))


class Chunker(ABC):
    """Abstract class for chunking a datum into smaller pieces."""

    @abstractmethod
    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a datum into smaller pieces."""


class CodeFileChunker(Chunker):
    """Splits a code file into chunks of at most `max_tokens` tokens each."""

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.text_chunker = TextFileChunker(max_tokens)

    @staticmethod
    def _get_language_from_filename(filename: str):
        """Returns a canonical name for the language of the file, based on its extension.
        Returns None if the language is unknown to the pygments lexer.
        """
        # pygments doesn't recognize .tsx files and returns None. So we need to special-case them.
        extension = os.path.splitext(filename)[1]
        if extension == ".tsx":
            return "tsx"

        try:
            lexer = pygments.lexers.get_lexer_for_filename(filename)
            return lexer.name.lower()
        except pygments.util.ClassNotFound:
            return None

    def _chunk_node(self, node: Node, file_content: str, file_metadata: Dict) -> List[FileChunk]:
        """Splits a node in the parse tree into a flat list of chunks."""
        node_chunk = FileChunk(file_content, file_metadata, node.start_byte, node.end_byte)

        if node_chunk.num_tokens <= self.max_tokens:
            return [node_chunk]

        if not node.children:
            # This is a leaf node, but it's too long. We'll have to split it with a text tokenizer.
            return self.text_chunker.chunk(file_content[node.start_byte : node.end_byte], file_metadata)

        chunks = []
        for child in node.children:
            chunks.extend(self._chunk_node(child, file_content, file_metadata))

        for chunk in chunks:
            # This should always be true. Otherwise there must be a bug in the code.
            assert chunk.num_tokens <= self.max_tokens

        # Merge neighboring chunks if their combined size doesn't exceed max_tokens. The goal is to avoid pathologically
        # small chunks that end up being undeservedly preferred by the retriever.
        merged_chunks = []
        for chunk in chunks:
            if not merged_chunks:
                merged_chunks.append(chunk)
            elif merged_chunks[-1].num_tokens + chunk.num_tokens < self.max_tokens - 50:
                # There's a good chance that merging these two chunks will be under the token limit. We're not 100% sure
                # at this point, because tokenization is not necessarily additive.
                merged = FileChunk(
                    file_content,
                    file_metadata,
                    merged_chunks[-1].start_byte,
                    chunk.end_byte,
                )
                if merged.num_tokens <= self.max_tokens:
                    merged_chunks[-1] = merged
                else:
                    merged_chunks.append(chunk)
            else:
                merged_chunks.append(chunk)
        chunks = merged_chunks

        for chunk in merged_chunks:
            # This should always be true. Otherwise there's a bug worth investigating.
            assert chunk.num_tokens <= self.max_tokens

        return merged_chunks

    @staticmethod
    def is_code_file(filename: str) -> bool:
        """Checks whether pygment & tree_sitter can parse the file as code."""
        language = CodeFileChunker._get_language_from_filename(filename)
        return language and language not in ["text only", "None"]

    @staticmethod
    def parse_tree(filename: str, content: str) -> List[str]:
        """Parses the code in a file and returns the parse tree."""
        language = CodeFileChunker._get_language_from_filename(filename)

        if not language or language in ["text only", "None"]:
            logging.debug("%s doesn't seem to be a code file.", filename)
            return None

        try:
            parser = get_parser(language)
        except LookupError:
            logging.debug("%s doesn't seem to be a code file.", filename)
            return None
        # This should never happen unless there's a bug in the code, but we'd rather not crash.
        except Exception as e:
            logging.warn("Failed to get parser for %s: %s", filename, e)
            return None

        tree = parser.parse(bytes(content, "utf8"))

        if not tree.root_node.children or tree.root_node.children[0].type == "ERROR":
            logging.warning("Failed to parse code in %s.", filename)
            return None
        return tree

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a code file into smaller pieces."""
        file_content = content
        file_metadata = metadata
        file_path = metadata["file_path"]

        if not file_content.strip():
            return []

        tree = self.parse_tree(file_path, file_content)
        if tree is None:
            return []

        file_chunks = self._chunk_node(tree.root_node, file_content, file_metadata)
        for chunk in file_chunks:
            # Make sure that the chunk has content and doesn't exceed the max_tokens limit. Otherwise there must be
            # a bug in the code.
            assert (
                chunk.num_tokens <= self.max_tokens
            ), f"Chunk size {chunk.num_tokens} exceeds max_tokens {self.max_tokens}."

        return file_chunks


class TextFileChunker(Chunker):
    """Wrapper around semchunk: https://github.com/umarbutler/semchunk."""

    def __init__(self, max_tokens: int):
        self.max_tokens = max_tokens
        self.count_tokens = lambda text: len(tokenizer.encode(text, disallowed_special=()))

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        """Chunks a text file into smaller pieces."""
        file_content = content
        file_metadata = metadata
        file_path = file_metadata["file_path"]

        # We need to allocate some tokens for the filename, which is part of the chunk content.
        extra_tokens = self.count_tokens(file_path + "\n\n")
        text_chunks = chunk_via_semchunk(file_content, self.max_tokens - extra_tokens, self.count_tokens)

        file_chunks = []
        start = 0
        for text_chunk in text_chunks:
            # This assertion should always be true. Otherwise there's a bug worth finding.
            assert self.count_tokens(text_chunk) <= self.max_tokens - extra_tokens

            # Find the start/end positions of the chunks.
            start = file_content.index(text_chunk, start)
            if start == -1:
                logging.warning("Couldn't find semchunk in content: %s", text_chunk)
            else:
                end = start + len(text_chunk)
                file_chunks.append(FileChunk(file_content, file_metadata, start, end))

            start = end

        return file_chunks


class IpynbFileChunker(Chunker):
    """Extracts the python code from a Jupyter notebook, removing all the boilerplate.

    Based on https://github.com/GoogleCloudPlatform/generative-ai/blob/main/language/code/code_retrieval_augmented_generation.ipynb
    """

    def __init__(self, code_chunker: CodeFileChunker):
        self.code_chunker = code_chunker

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        filename = metadata["file_path"]

        if not filename.lower().endswith(".ipynb"):
            logging.warn("IPYNBChunker is only for .ipynb files.")
            return []

        notebook = nbformat.reads(content, as_version=nbformat.NO_CONVERT)
        python_code = "\n".join([cell.source for cell in notebook.cells if cell.cell_type == "code"])

        tmp_metadata = {"file_path": filename.replace(".ipynb", ".py")}
        chunks = self.code_chunker.chunk(python_code, tmp_metadata)

        for chunk in chunks:
            # Update filenames back to .ipynb
            chunk.metadata["file_path"] = filename
        return chunks


In [542]:
class UniversalFileChunker(Chunker):
    """Chunks a file into smaller pieces, regardless of whether it's code or text."""

    def __init__(self, max_tokens: int):
        print('here')
        self.max_tokens = max_tokens
        self.code_chunker = CodeFileChunker(max_tokens)
        self.ipynb_chunker = IpynbFileChunker(self.code_chunker)
        self.text_chunker = TextFileChunker(max_tokens)

    def chunk(self, content: Any, metadata: Dict) -> List[Chunk]:
        if not "file_path" in metadata:
            raise ValueError("metadata must contain a 'file_path' key.")
        file_path = metadata["file_path"]
        
        # Figure out the appropriate chunker to use.
        if file_path.lower().endswith(".ipynb"):
            chunker = self.ipynb_chunker
        elif CodeFileChunker.is_code_file(file_path):
            chunker = self.code_chunker
        else:
            chunker = self.text_chunker
        chunk = chunker.chunk(content, metadata)
        
        return chunk
chunk = UniversalFileChunker(4000)

chunk.chunk("export const fles = () => {console.log('nothing')}", {'file_path': 'filename.ts'})

here


[FileChunk(file_content="export const fles = () => {console.log('nothing')}", file_metadata={'file_path': 'filename.ts'}, start_byte=0, end_byte=50)]

In [1]:
! pip install langchain-text-splitters
! pip install langchain_experimental langchain_openai

Note: you may need to restart the kernel to use updated packages.


In [125]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    RecursiveJsonSplitter
)
import json

class BaseChunkingStrategy:
    """Base class with shared chunk size, overlap, and a default splitter."""
    chunk_size = 1024
    chunk_overlap = 200

    @classmethod
    def splitter(cls):
        """Lazily initialize and return the default splitter."""
        return CharacterTextSplitter.from_tiktoken_encoder(
            encoding="cl100k_base",
            chunk_size=cls.chunk_size,
            chunk_overlap=cls.chunk_overlap,
            add_start_index=True
        )

    @classmethod
    def create_documents(cls, doc):
        content = doc.page_content
        metadata = doc.metadata
        """Split text using the default splitter."""
        return cls.splitter().create_documents(
            [content], 
            [metadata])


class SemanticChunkingStrategy(BaseChunkingStrategy):
    """Semantic chunking strategy with a shared splitter."""
    def __init__(self):
        self.splitter = SemanticChunker(
            OpenAIEmbeddings(),
            breakpoint_threshold_type="percentile",
            add_start_index=True
        )

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content], 
                [metadata])
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(
                [content], 
                [metadata])


class CodeChunkingStrategy(BaseChunkingStrategy):
    """Code chunking strategy with instance-specific language."""
    def __init__(self, language=None):
        self.language = language
        self.splitter = RecursiveCharacterTextSplitter.from_language(
            language=self.language,
            chunk_size=BaseChunkingStrategy.chunk_size,
            chunk_overlap=BaseChunkingStrategy.chunk_overlap,
            add_start_index=True
        )

    def create_documents(self, doc):
        content = doc.page_content
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [content], 
                [metadata])
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents(
                [content], 
                [metadata])


class JsonChunkingStrategy(BaseChunkingStrategy):
    """JSON chunking strategy with a shared splitter."""
    
    def __init__(self):
        self.splitter = RecursiveJsonSplitter(
            max_chunk_size=MAX_TOKENS_CHUNK_SIZE,
        )

    def create_documents(self, doc):
        content = doc.page_content
        json_data = json.loads(content)
        metadata = doc.metadata
        try:
            return self.splitter.create_documents(
                [json_data], 
                [metadata], 
                convert_lists=True)
        except Exception:
            # Fallback to default splitter
            return BaseChunkingStrategy.create_documents([content], [metadata])

In [120]:
PYTHON_CODE = """
class Task:
    def __init__(self, description, completed=False):
        self.description = description
        self.completed = completed

    def __str__(self):
        status = "✔" if self.completed else "✘"
        return f"{status} {self.description}"

class TaskTracker:
    def __init__(self):
        self.tasks = []

    def add_task(self, description):
        task = Task(description)
        self.tasks.append(task)
        print(f"Task added: {description}")

    def remove_task(self, index):
        if 0 <= index < len(self.tasks):
            removed_task = self.tasks.pop(index)
            print(f"Task removed: {removed_task.description}")
        else:
            print("Invalid task number.")

    def list_tasks(self):
        if not self.tasks:
            print("No tasks in the list.")
        else:
            print("Tasks:")
            for i, task in enumerate(self.tasks):
                print(f"{i + 1}. {task}")

    def mark_complete(self, index):
        if 0 <= index < len(self.tasks):
            self.tasks[index].completed = True
            print(f"Task marked as complete: {self.tasks[index].description}")
        else:
            print("Invalid task number.")

def display_menu():
    print("\nTask Tracker")
    print("1. Add Task")
    print("2. Remove Task")
    print("3. List Tasks")
    print("4. Mark Task Complete")
    print("5. Exit")

def main():
    tracker = TaskTracker()
    while True:
        display_menu()
        choice = input("Enter your choice (1-5): ").strip()
        if choice == "1":
            description = input("Enter task description: ").strip()
            tracker.add_task(description)
        elif choice == "2":
            tracker.list_tasks()
            try:
                index = int(input("Enter task number to remove: ")) - 1
                tracker.remove_task(index)
            except ValueError:
                print("Invalid input. Please enter a number.")
        elif choice == "3":
            tracker.list_tasks()
        elif choice == "4":
            tracker.list_tasks()
            try:
                index = int(input("Enter task number to mark as complete: ")) - 1
                tracker.mark_complete(index)
            except ValueError:
                print("Invalid input. Please enter a number.")
        elif choice == "5":
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()

"""

chunker = CodeChunkingStrategy('python')

filepath = '/Users/it/Desktop/Proj_CodebaseRag2/notebook/SecureAgent/src/app.ts'
with open(filepath, "r", encoding='utf-8') as f:
    contents = f.read()
file = files[0]
chunked_file = chunker.create_documents(file)
chunked_file


[Document(metadata={'filename': 'app.ts', 'path': 'SecureAgent/src/app.ts', 'url': 'https://www.github.com/CoderAgent/SecureAgent/blob/main/src/app.ts', 'start_index': 0}, page_content='import { Octokit } from "@octokit/rest";\nimport { createNodeMiddleware } from "@octokit/webhooks";\nimport { WebhookEventMap } from "@octokit/webhooks-definitions/schema";\nimport * as http from "http";\nimport { App } from "octokit";\nimport { Review } from "./constants";\nimport { env } from "./env";\nimport { processPullRequest } from "./review-agent";\nimport { applyReview } from "./reviews";\n\n// This creates a new instance of the Octokit App class.\nconst reviewApp = new App({\n  appId: env.GITHUB_APP_ID,\n  privateKey: env.GITHUB_PRIVATE_KEY,\n  webhooks: {\n    secret: env.GITHUB_WEBHOOK_SECRET,\n  },\n});'),
 Document(metadata={'filename': 'app.ts', 'path': 'SecureAgent/src/app.ts', 'url': 'https://www.github.com/CoderAgent/SecureAgent/blob/main/src/app.ts', 'start_index': 608}, page_content=

In [126]:
data = {
    "name": "John Doe",
    "age": 30,
    "is_employee": True,
    "skills": ["Python", "JavaScript", "SQL"],
    "address": {
        "street": "123 Main St",
        "city": "New York",
        "zip": "10001"
    }
}

chunker = JsonChunkingStrategy()
chunked_file = chunker.create_documents(data)

AttributeError: 'dict' object has no attribute 'page_content'

# Embedding


In [ ]:
def get_embeddings(text, model):
    model = 

# GRAPH RAG

# Storing in the the vectorstore

# Retrieval and reranking using nvidia